In [ ]:
# ── Imports ──

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import re
from datetime import datetime
from scipy.stats import fisher_exact
import matplotlib.colors as mcolors

In [ ]:
# ── Paths and configuration ──

BASE_DIR    = os.path.dirname(os.path.abspath("__file__"))
OUTPUT_DIR  = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATHS = {
    # Inputs from earlier stages — all in outputs subdirectory
    "diff_abundance"   : os.path.join(OUTPUT_DIR, "differential_abundance_results.csv"),
    "feature_imp"      : os.path.join(OUTPUT_DIR, "feature_importances.csv"),
    "taxonomy"         : os.path.join(OUTPUT_DIR, "taxonomy_table.csv"),

    # Frolova annotation database — sits directly in BASE_DIR
    "frolova"          : os.path.join(BASE_DIR, "Table 1.XLSX"),

    # Stage 6 outputs — also go into outputs subdirectory
    "out_scfa"         : os.path.join(OUTPUT_DIR, "scfa_annotation_table.csv"),
    "out_summary_md"   : os.path.join(OUTPUT_DIR, "nutrigenomic_pathway_summary.md"),
    "out_barplot"      : os.path.join(OUTPUT_DIR, "fig_scfa_barplot.png"),
    "out_heatmap"      : os.path.join(OUTPUT_DIR, "fig_scfa_heatmap.png"),
    "out_report"       : os.path.join(OUTPUT_DIR, "stage6_report.txt"),
}

# Fiber group labels (must match Stage 5 output)
FIBER_GROUP_LABELS = {
    1: "Prebiotic oligosaccharides",
    2: "Resistant starch",
    3: "Soluble fiber",
    4: "Grain/wheat fiber",
    5: "Mixed/unspecified",
}

SCFA_TYPES = ["butyrate", "propionate", "acetate", "formate", "lactate"]

print("=" * 70)
print("Stage 6 — Nutrigenomic Pathway Layer")
print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# Verify all input files exist before proceeding
missing = [k for k, v in PATHS.items() if not k.startswith("out") and not os.path.exists(v)]
if missing:
    raise FileNotFoundError(
        f"Missing input files: {missing}\n"
        f"Expected locations:\n" +
        "\n".join(f"  {k}: {PATHS[k]}" for k in missing)
    )
print("\nAll input files found.")

In [ ]:
# ── GTDB → NCBI genus mapping ──

GTDB_TO_NCBI_GENUS = {
    # Bacteroidota renames
    "Phocaeicola"          : "Bacteroides",
    "Prevotella_B"         : "Prevotella",
    "Prevotella_C"         : "Prevotella",
    "Alistipes_A"          : "Alistipes",
    "Alistipes_B"          : "Alistipes",
    "Parabacteroides_A"    : "Parabacteroides",

    # Firmicutes / Lachnospiraceae renames
    "Agathobacter"         : "Roseburia",
    "Mediterraneibacter"   : "Ruminococcus",
    "Blautia_A"            : "Blautia",
    "Blautia_B"            : "Blautia",
    "Lachnospira_A"        : "Lachnospira",
    "Roseburia_A"          : "Roseburia",
    "Butyrivibrio_A"       : "Butyrivibrio",
    "Coprococcus_A"        : "Coprococcus",
    "Coprococcus_B"        : "Coprococcus",
    "Anaerostipes_A"       : "Anaerostipes",

    # Firmicutes / Ruminococcaceae renames
    "Ruminococcus_A"       : "Ruminococcus",
    "Ruminococcus_B"       : "Ruminococcus",
    "Ruminococcus_C"       : "Ruminococcus",
    "Ruminococcus_D"       : "Ruminococcus",
    "Ruminococcus_E"       : "Ruminococcus",
    "Faecalibacterium_A"   : "Faecalibacterium",
    "Faecalibacterium_B"   : "Faecalibacterium",
    "Faecalibacterium_C"   : "Faecalibacterium",
    "Faecalibacterium_D"   : "Faecalibacterium",
    "Faecalibacterium_E"   : "Faecalibacterium",

    # Actinobacteriota
    "Bifidobacterium_A"    : "Bifidobacterium",

    # Verrucomicrobiota
    "Akkermansia_A"        : "Akkermansia",
}

# Species-level overrides — for cases where the genus maps correctly
# but the species epithet itself was renamed in GTDB r95
GTDB_TO_NCBI_SPECIES = {
    "Phocaeicola dorei"          : "Bacteroides dorei",
    "Phocaeicola vulgatus"       : "Bacteroides vulgatus",
    "Phocaeicola massiliensis"   : "Bacteroides massiliensis",
    "Agathobacter rectalis"      : "Roseburia intestinalis",
    "Mediterraneibacter gnavus"  : "Ruminococcus gnavus",
    "Mediterraneibacter torques" : "Ruminococcus torques",
}

print(f"GTDB→NCBI genus mapping loaded:    {len(GTDB_TO_NCBI_GENUS)} entries")
print(f"GTDB→NCBI species overrides loaded: {len(GTDB_TO_NCBI_SPECIES)} entries")

In [ ]:
# ── HDAC / downstream gene linkage table ──

HDAC_LINKAGE = {
    "butyrate": {
        "mechanism"    : "HDAC1/2/3 inhibition (class I HDACs)",
        "receptors"    : ["GPR109A", "GPR41"],
        "target_genes" : ["FOXP3", "IL-10", "GPR109A", "TGFB1", "CDKN1A"],
        "phenotype"    : "Treg induction, anti-inflammatory, colonocyte energy substrate, anti-tumorigenic",
        "references"   : "Donohoe 2012; Furusawa 2013; Thangaraju 2009",
    },
    "propionate": {
        "mechanism"    : "Partial HDAC inhibition + GPR41/GPR43 agonism",
        "receptors"    : ["GPR41", "GPR43"],
        "target_genes" : ["GCG", "PYY", "FFAR3", "FFAR2"],
        "phenotype"    : "GLP-1 and PYY secretion, appetite regulation, gut motility",
        "references"   : "Chambers 2018; Samuel 2008",
    },
    "acetate": {
        "mechanism"    : "GPR43 agonism; weak HDAC inhibition at physiological concentrations",
        "receptors"    : ["GPR43", "GPR41"],
        "target_genes" : ["FFAR2", "IL-18", "NLRP3"],
        "phenotype"    : "Neutrophil/macrophage activation, intestinal barrier integrity, immune modulation",
        "references"   : "Maslowski 2009; Macia 2015",
    },
    "formate": {
        "mechanism"    : "One-carbon metabolism; indirect effects via folate cycle",
        "receptors"    : [],
        "target_genes" : ["MTHFR", "MTR"],
        "phenotype"    : "Methylation substrate, mitochondrial function support",
        "references"   : "Koh 2016",
    },
    "lactate": {
        "mechanism"    : "Cross-feeding substrate for butyrate producers; GPR81 agonism",
        "receptors"    : ["GPR81"],
        "target_genes" : ["HCAR1"],
        "phenotype"    : "Indirect butyrate boost via cross-feeding; anti-lipolytic signalling",
        "references"   : "Schieber & Chandel 2014; Belenguer 2006",
    },
}

print("HDAC/pathway linkage table loaded for:", list(HDAC_LINKAGE.keys()))


In [ ]:
# ── Load Frolova et al. 2022 database ──

print("\n--- Loading Frolova et al. 2022 (Table 1.XLSX) ---")

xl = pd.ExcelFile(PATHS["frolova"])
print(f"Sheets found: {xl.sheet_names}")

# --- Taxonomy sheet ---
# Row 0 = sheet title string, Row 1 = actual column headers
tax_raw = xl.parse("Taxonomy", header=None)
tax_raw.columns = tax_raw.iloc[1]
tax_frolova = tax_raw.iloc[2:].reset_index(drop=True)
tax_frolova.columns = tax_frolova.columns.str.strip()
tax_frolova = tax_frolova[["Genome id", "Genus", "Species"]].copy()
tax_frolova["Genome id"] = tax_frolova["Genome id"].astype(str).str.strip()
tax_frolova["Genus"]     = tax_frolova["Genus"].astype(str).str.strip()
tax_frolova["Species"]   = tax_frolova["Species"].astype(str).str.strip()
print(f"Taxonomy sheet: {len(tax_frolova)} genomes")


def parse_frolova_sheet(sheet_name, binary_col_name):
    """
    Parse a Frolova pathway sheet (Butyrate or Propionate).
    Row 0 = sheet title, Row 1 = actual header.
    Returns DataFrame with: genome_id, binary_col_name, pathway_variant (if present).
    """
    raw = xl.parse(sheet_name, header=None)
    raw.columns = raw.iloc[1]
    df = raw.iloc[2:].reset_index(drop=True)
    df.columns = df.columns.astype(str).str.strip()

    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if "genome id" in cl:
            col_map[c] = "genome_id"
        elif "variant" in cl:
            col_map[c] = "pathway_variant"
        elif "binary phenotype" in cl:
            # Exclude AFL-specific phenotype columns
            if not any(x in cl for x in ["acetate","formate","lactate","l-lactate","d-lactate"]):
                col_map[c] = binary_col_name

    df = df.rename(columns=col_map)

    keep = ["genome_id", binary_col_name]
    if "pathway_variant" in df.columns:
        keep.append("pathway_variant")
    # Only keep columns that actually exist after renaming
    keep = [c for c in keep if c in df.columns]

    df = df[keep].copy()
    df["genome_id"]       = df["genome_id"].astype(str).str.strip()
    df[binary_col_name]   = pd.to_numeric(df[binary_col_name], errors="coerce").fillna(0).astype(int)

    return df


# --- Butyrate sheet ---
but_df = parse_frolova_sheet("Butyrate", "butyrate")
but_df = but_df.rename(columns={"pathway_variant": "butyrate_variant"})
print(f"Butyrate sheet:    {len(but_df)} genomes, "
      f"{but_df['butyrate'].sum()} producers")

# --- Propionate sheet ---
pro_df = parse_frolova_sheet("Propionate", "propionate")
pro_df = pro_df.rename(columns={"pathway_variant": "propionate_variant"})
print(f"Propionate sheet:  {len(pro_df)} genomes, "
      f"{pro_df['propionate'].sum()} producers")

# --- Acetate-Formate-Lactate sheet ---
afl_raw = xl.parse("Acetate-Formate-Lactate", header=None)
afl_raw.columns = afl_raw.iloc[1]
afl_df = afl_raw.iloc[2:].reset_index(drop=True)
afl_df.columns = afl_df.columns.astype(str).str.strip()

afl_col_map = {}
for c in afl_df.columns:
    cl = c.lower()
    if "genome id" in cl:
        afl_col_map[c] = "genome_id"
    elif "acetate binary" in cl:
        afl_col_map[c] = "acetate"
    elif "formate binary" in cl:
        afl_col_map[c] = "formate"
    elif "l-lactate binary" in cl:
        afl_col_map[c] = "lactate_L"
    elif "d-lactate binary" in cl:
        afl_col_map[c] = "lactate_D"
    elif "variant" in cl:
        afl_col_map[c] = "afl_variant"

afl_df = afl_df.rename(columns=afl_col_map)
afl_keep = [c for c in ["genome_id","acetate","formate","lactate_L","lactate_D","afl_variant"]
            if c in afl_df.columns]
afl_df = afl_df[afl_keep].copy()
afl_df["genome_id"] = afl_df["genome_id"].astype(str).str.strip()

for col in ["acetate","formate","lactate_L","lactate_D"]:
    if col in afl_df.columns:
        afl_df[col] = pd.to_numeric(afl_df[col], errors="coerce").fillna(0).astype(int)

# Collapse L-lactate and D-lactate → single lactate column (either = producer)
if "lactate_L" in afl_df.columns and "lactate_D" in afl_df.columns:
    afl_df["lactate"] = ((afl_df["lactate_L"] == 1) | (afl_df["lactate_D"] == 1)).astype(int)
    afl_df = afl_df.drop(columns=["lactate_L","lactate_D"])

print(f"AFL sheet:         {len(afl_df)} genomes, "
      f"acetate={afl_df['acetate'].sum()}, "
      f"formate={afl_df['formate'].sum()}, "
      f"lactate={afl_df['lactate'].sum()}")

# --- Merge all sheets into one Frolova master table ---
frolova = tax_frolova.rename(columns={
    "Genome id" : "genome_id",
    "Genus"     : "ncbi_genus",
    "Species"   : "ncbi_species",
})

frolova = frolova.merge(
    but_df[["genome_id","butyrate","butyrate_variant"]] if "butyrate_variant" in but_df.columns
    else but_df[["genome_id","butyrate"]],
    on="genome_id", how="left"
)
frolova = frolova.merge(
    pro_df[["genome_id","propionate","propionate_variant"]] if "propionate_variant" in pro_df.columns
    else pro_df[["genome_id","propionate"]],
    on="genome_id", how="left"
)

afl_merge_cols = ["genome_id","acetate","formate","lactate"]
if "afl_variant" in afl_df.columns:
    afl_merge_cols.append("afl_variant")
frolova = frolova.merge(afl_df[afl_merge_cols], on="genome_id", how="left")

# Fill NaN producer flags with 0
for col in SCFA_TYPES:
    if col in frolova.columns:
        frolova[col] = frolova[col].fillna(0).astype(int)

print(f"\nFrolova master table: {len(frolova)} genomes, "
      f"{frolova['ncbi_genus'].nunique()} unique genera")
print(f"Columns: {frolova.columns.tolist()}")


In [ ]:
# ── Genus-level SCFA lookup ──

genus_groups = frolova.groupby("ncbi_genus")
genus_lookup = {}

for genus, grp in genus_groups:
    entry = {"n_genomes": len(grp)}
    for scfa in SCFA_TYPES:
        if scfa in grp.columns:
            prop = grp[scfa].mean()
            entry[scfa]                 = int(prop > 0)   # any-member threshold
            entry[f"{scfa}_proportion"] = round(prop, 3)
        else:
            entry[scfa]                 = 0
            entry[f"{scfa}_proportion"] = 0.0
    genus_lookup[genus] = entry

print(f"Genus-level lookup built: {len(genus_lookup)} genera")
print()

# Spot-check key gut taxa
for test_genus in ["Faecalibacterium", "Bifidobacterium", "Roseburia",
                   "Bacteroides", "Akkermansia", "Blautia"]:
    entry = genus_lookup.get(test_genus)
    if entry:
        producers = {s: entry[s] for s in SCFA_TYPES}
        props     = {s: entry[f"{s}_proportion"] for s in SCFA_TYPES}
        print(f"{test_genus} (n={entry['n_genomes']} genomes)")
        print(f"  Binary:      {producers}")
        print(f"  Proportions: {props}")
    else:
        print(f"{test_genus}: NOT FOUND in Frolova")
    print()

In [ ]:
# ── SCFA enrichment test (Fisher's exact) ──


# --- 1. Load the three input tables ---
tax_full = pd.read_csv(os.path.join(OUTPUT_DIR, "taxonomy_table.csv"))
delta_fi = pd.read_csv(os.path.join(OUTPUT_DIR, "delta_feature_importances.csv"))
lme_sig  = pd.read_csv(os.path.join(OUTPUT_DIR, "lme_significant_otus_annotated.csv"))

filtered_otu_ids = set(delta_fi["OTU_ID"].values)
lme_sig_ids      = set(lme_sig["OTU_ID"].values)

print(f"Full taxonomy table:     {len(tax_full)} OTUs")
print(f"Prevalence-filtered:     {len(filtered_otu_ids)} OTUs")
print(f"LME-significant:         {len(lme_sig_ids)} OTUs")

# --- 2. Filter taxonomy to the 9,612 OTUs ---
tax = tax_full[tax_full["OTU_ID"].isin(filtered_otu_ids)].copy()
print(f"Taxonomy after filter:   {len(tax)} OTUs")

# --- 3. Parse GTDB genus from taxonomy string ---
def extract_gtdb_genus(tax_string):
    """Extract genus from GTDB taxonomy string, return None if absent."""
    if pd.isna(tax_string):
        return None
    for field in tax_string.split(";"):
        field = field.strip()
        if field.startswith("g__"):
            g = field[3:].strip()
            return g if g else None
    return None

tax["gtdb_genus"] = tax["taxonomy"].apply(extract_gtdb_genus)

# --- 4. Map GTDB genus → NCBI genus (using notebook 06 dictionaries) ---
tax["ncbi_genus"] = tax["gtdb_genus"].map(
    lambda g: GTDB_TO_NCBI_GENUS.get(g, g) if g else None
)

# --- 5. Look up SCFA producer status from genus_lookup ---
def lookup_scfa(ncbi_genus):
    """Return (butyrate, propionate, acetate) binary flags from genus_lookup."""
    if ncbi_genus is None or ncbi_genus == "":
        return 0, 0, 0
    entry = genus_lookup.get(ncbi_genus)
    if entry is None:
        return 0, 0, 0
    return entry.get("butyrate", 0), entry.get("propionate", 0), entry.get("acetate", 0)

tax[["butyrate", "propionate", "acetate"]] = tax["ncbi_genus"].apply(
    lambda g: pd.Series(lookup_scfa(g))
)

# Any SCFA = butyrate OR propionate OR acetate
tax["any_scfa"] = ((tax["butyrate"] == 1) | 
                   (tax["propionate"] == 1) | 
                   (tax["acetate"] == 1)).astype(int)

# Flag LME-significant
tax["lme_significant"] = tax["OTU_ID"].isin(lme_sig_ids).astype(int)

# --- 6. Build 2×2 tables and run Fisher's exact test ---
results = []

for label, col in [("any_scfa", "any_scfa"), ("butyrate_only", "butyrate")]:
    sig_prod     = int(tax.loc[(tax["lme_significant"] == 1) & (tax[col] == 1)].shape[0])
    sig_nonprod  = int(tax.loc[(tax["lme_significant"] == 1) & (tax[col] == 0)].shape[0])
    bg_prod      = int(tax.loc[(tax["lme_significant"] == 0) & (tax[col] == 1)].shape[0])
    bg_nonprod   = int(tax.loc[(tax["lme_significant"] == 0) & (tax[col] == 0)].shape[0])
    
    table = [[sig_prod, sig_nonprod],
             [bg_prod,  bg_nonprod]]
    
    odds_ratio, p_value = fisher_exact(table, alternative="greater")
    
    results.append({
        "test": label,
        "sig_producer": sig_prod,
        "sig_nonproducer": sig_nonprod,
        "bg_producer": bg_prod,
        "bg_nonproducer": bg_nonprod,
        "odds_ratio": round(odds_ratio, 3),
        "p_value": p_value,
    })
    
    print(f"\n{'='*60}")
    print(f"Fisher's exact test: {label}")
    print(f"{'='*60}")
    print(f"                    Producer  Non-producer   Total")
    print(f"  LME-significant   {sig_prod:>8}  {sig_nonprod:>12}   {sig_prod + sig_nonprod}")
    print(f"  Background        {bg_prod:>8}  {bg_nonprod:>12}   {bg_prod + bg_nonprod}")
    print(f"  Odds ratio: {odds_ratio:.3f}")
    print(f"  p-value:    {p_value:.4e}")

# --- 7. Summary counts for sanity check ---
n_with_genus  = tax["ncbi_genus"].notna().sum()
n_in_frolova  = tax["ncbi_genus"].apply(lambda g: g in genus_lookup if g else False).sum()
n_novel       = tax["gtdb_genus"].apply(
    lambda g: bool(g) and any(g.startswith(p) for p in ["UBA", "CAG", "UMGS", "GCA-", "GUT"])
).sum()

print(f"\n{'='*60}")
print(f"Annotation coverage (9,612 OTUs)")
print(f"{'='*60}")
print(f"  With GTDB genus:           {n_with_genus}")
print(f"  Mapped to Frolova genus:   {n_in_frolova}")
print(f"  Novel placeholder genera:  {n_novel}")
print(f"  any_scfa producers:        {tax['any_scfa'].sum()}")
print(f"  butyrate producers:        {tax['butyrate'].sum()}")

# --- 8. Save results ---
results_df = pd.DataFrame(results)
out_path = os.path.join(OUTPUT_DIR, "scfa_enrichment_results.csv")
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

In [ ]:
# ── Fisher's exact (Frolova-annotatable universe) ──

# Filter to OTUs whose NCBI genus exists in genus_lookup
tax_annotatable = tax[
    tax["ncbi_genus"].apply(lambda g: g in genus_lookup if g else False)
].copy()

n_total = len(tax_annotatable)
n_sig   = tax_annotatable["lme_significant"].sum()
n_bg    = n_total - n_sig

print(f"Annotatable universe: {n_total} OTUs ({n_sig} LME-significant, {n_bg} background)")

results_a = []

for label, col in [("any_scfa", "any_scfa"), ("butyrate_only", "butyrate")]:
    sig_prod    = int(tax_annotatable.loc[(tax_annotatable["lme_significant"] == 1) & (tax_annotatable[col] == 1)].shape[0])
    sig_nonprod = int(tax_annotatable.loc[(tax_annotatable["lme_significant"] == 1) & (tax_annotatable[col] == 0)].shape[0])
    bg_prod     = int(tax_annotatable.loc[(tax_annotatable["lme_significant"] == 0) & (tax_annotatable[col] == 1)].shape[0])
    bg_nonprod  = int(tax_annotatable.loc[(tax_annotatable["lme_significant"] == 0) & (tax_annotatable[col] == 0)].shape[0])

    table = [[sig_prod, sig_nonprod],
             [bg_prod,  bg_nonprod]]

    odds_ratio, p_value = fisher_exact(table, alternative="greater")

    results_a.append({
        "test": label + "_annotatable",
        "sig_producer": sig_prod,
        "sig_nonproducer": sig_nonprod,
        "bg_producer": bg_prod,
        "bg_nonproducer": bg_nonprod,
        "odds_ratio": round(odds_ratio, 3),
        "p_value": p_value,
    })

    bg_rate  = bg_prod / (bg_prod + bg_nonprod) * 100
    sig_rate = sig_prod / (sig_prod + sig_nonprod) * 100

    print(f"\n{'='*60}")
    print(f"Fisher's exact test: {label} (annotatable universe)")
    print(f"{'='*60}")
    print(f"                    Producer  Non-producer   Total   Rate")
    print(f"  LME-significant   {sig_prod:>8}  {sig_nonprod:>12}   {sig_prod + sig_nonprod:>5}   {sig_rate:.1f}%")
    print(f"  Background        {bg_prod:>8}  {bg_nonprod:>12}   {bg_prod + bg_nonprod:>5}   {bg_rate:.1f}%")
    print(f"  Odds ratio: {odds_ratio:.3f}")
    print(f"  p-value:    {p_value:.4e}")

# Append to results and save
results_a_df = pd.DataFrame(results_a)
results_all  = pd.concat([results_df, results_a_df], ignore_index=True)
results_all.to_csv(os.path.join(OUTPUT_DIR, "scfa_enrichment_results.csv"), index=False)
print(f"\nUpdated: {os.path.join(OUTPUT_DIR, 'scfa_enrichment_results.csv')}")

In [ ]:
# ── Load Stage 3 and Stage 5 inputs ──

print("\n--- Loading Stage 5 differential abundance results ---")
diff_ab = pd.read_csv(PATHS["diff_abundance"])
print(f"Loaded: {diff_ab.shape}")
print(f"Columns: {diff_ab.columns.tolist()}")

# Expected columns from Stage 5:
# OTU_ID, fiber_group, log2FoldChange, padj, direction
# If column names differ, update the remap dictionary below
DIFF_AB_REMAP = {
    "clr_diff" : "log2FoldChange",
}
if DIFF_AB_REMAP:
    diff_ab = diff_ab.rename(columns=DIFF_AB_REMAP)

required_diff = ["OTU_ID", "fiber_group", "log2FoldChange", "padj"]
missing_diff  = [c for c in required_diff if c not in diff_ab.columns]
if missing_diff:
    raise ValueError(
        f"differential_abundance_results.csv missing columns: {missing_diff}\n"
        f"Actual columns: {diff_ab.columns.tolist()}\n"
        "Add entries to DIFF_AB_REMAP above to fix column name mismatches."
    )

# Add direction column if Stage 5 didn't produce one
if "direction" not in diff_ab.columns:
    diff_ab["direction"] = diff_ab["log2FoldChange"].apply(
        lambda x: "up" if x > 0 else "down"
    )

print(f"\nStage 5 OTUs loaded:      {len(diff_ab)}")
print(f"Fiber groups present:     {sorted(diff_ab['fiber_group'].unique())}")
print(f"Up (fiber-enriched):      {(diff_ab['direction']=='up').sum()}")
print(f"Down (fiber-depleted):    {(diff_ab['direction']=='down').sum()}")

# ---------------------------------------------------------------------------

print("\n--- Loading Stage 3 feature importances ---")
feat_imp = pd.read_csv(PATHS["feature_imp"])
print(f"Loaded: {feat_imp.shape}")
print(f"Columns: {feat_imp.columns.tolist()}")

# Normalise OTU_ID column name
if "OTU_ID" not in feat_imp.columns:
    for candidate in ["otu_id", "feature", "index", "OTU"]:
        if candidate in feat_imp.columns:
            feat_imp = feat_imp.rename(columns={candidate: "OTU_ID"})
            break

# Find the importance/SHAP score column
shap_col = None
for candidate in ["mean_shap", "shap_value", "shap_importance",
                  "importance", "importance_score", "MeanDecreaseImpurity",
                  "MDI"]:
    if candidate in feat_imp.columns:
        shap_col = candidate
        break
if shap_col is None:
    raise ValueError(
        f"Cannot find SHAP/importance column in feature_importances.csv.\n"
        f"Columns present: {feat_imp.columns.tolist()}\n"
        "Add the correct column name to the candidate list above."
    )

feat_imp = (feat_imp
            .nlargest(50, shap_col)[["OTU_ID", shap_col]]
            .reset_index(drop=True)
            .rename(columns={shap_col: "shap_importance"}))

print(f"\nTop 50 SHAP OTUs loaded.")
print(f"Importance score range:   "
      f"{feat_imp['shap_importance'].min():.4f} – "
      f"{feat_imp['shap_importance'].max():.4f}")

# ---------------------------------------------------------------------------

print("\n--- Loading taxonomy table ---")
tax = pd.read_csv(PATHS["taxonomy"])
print(f"Loaded: {tax.shape}")
print(f"Columns: {tax.columns.tolist()}")

# Normalise column names
if "OTU_ID" not in tax.columns:
    tax = tax.rename(columns={tax.columns[0]: "OTU_ID"})
if "taxonomy" not in tax.columns:
    for candidate in ["Taxon", "taxon", "taxonomy_string",
                      "tax_string", "Taxonomy"]:
        if candidate in tax.columns:
            tax = tax.rename(columns={candidate: "taxonomy"})
            break

if "taxonomy" not in tax.columns:
    raise ValueError(
        f"Cannot find taxonomy string column in taxonomy_table.csv.\n"
        f"Columns present: {tax.columns.tolist()}"
    )

print(f"\nTaxonomy table: {len(tax)} OTUs")
print(f"Example taxonomy string:\n  {tax['taxonomy'].iloc[0]}")

In [ ]:
# ── Parse GTDB taxonomy ──

def parse_gtdb_taxonomy(tax_string):
    """
    Parse a GTDB taxonomy string and return (gtdb_genus, gtdb_species).
    Returns ("unknown", "unknown") if fields are absent or empty.
    """
    if not isinstance(tax_string, str):
        return "unknown", "unknown"

    genus, species = "unknown", "unknown"

    g_match = re.search(r'g__([^;]+)', tax_string)
    s_match = re.search(r's__([^;]+)', tax_string)

    if g_match:
        genus = g_match.group(1).strip()
        if genus == "":
            genus = "unknown"

    if s_match:
        species = s_match.group(1).strip()
        if species == "":
            species = "unknown"

    return genus, species


def resolve_ncbi_name(gtdb_genus, gtdb_species):
    """
    Apply GTDB→NCBI mapping to get the lookup name for Frolova.
    Returns (ncbi_genus, ncbi_species, mapping_note).
    """
    # Species-level override takes priority
    if gtdb_species != "unknown" and gtdb_species in GTDB_TO_NCBI_SPECIES:
        ncbi_sp  = GTDB_TO_NCBI_SPECIES[gtdb_species]
        ncbi_gen = ncbi_sp.split()[0]
        return ncbi_gen, ncbi_sp, "species_override"

    # Genus-level mapping
    if gtdb_genus in GTDB_TO_NCBI_GENUS:
        ncbi_gen = GTDB_TO_NCBI_GENUS[gtdb_genus]
        if gtdb_species != "unknown" and " " in gtdb_species:
            epithet  = gtdb_species.split(" ", 1)[1]
            ncbi_sp  = f"{ncbi_gen} {epithet}"
        else:
            ncbi_sp  = gtdb_species
        return ncbi_gen, ncbi_sp, "genus_mapped"

    # No mapping needed
    return gtdb_genus, gtdb_species, "direct"


# Parse taxonomy for all OTUs appearing in Stage 3 or Stage 5
all_otu_ids  = set(diff_ab["OTU_ID"].tolist()) | set(feat_imp["OTU_ID"].tolist())
tax_subset   = tax[tax["OTU_ID"].isin(all_otu_ids)].copy()

print(f"OTUs in scope (Stage 3 + Stage 5 union): {len(all_otu_ids)}")
print(f"OTUs found in taxonomy table:             {len(tax_subset)}")
missing_tax = all_otu_ids - set(tax_subset["OTU_ID"])
if missing_tax:
    print(f"WARNING: {len(missing_tax)} OTUs not found in taxonomy table")

# Parse GTDB strings
parsed = tax_subset["taxonomy"].apply(parse_gtdb_taxonomy)
tax_subset = tax_subset.copy()
tax_subset["gtdb_genus"]   = [p[0] for p in parsed]
tax_subset["gtdb_species"] = [p[1] for p in parsed]

# Resolve NCBI names
resolved = tax_subset.apply(
    lambda row: resolve_ncbi_name(row["gtdb_genus"], row["gtdb_species"]),
    axis=1
)
tax_subset["ncbi_genus"]   = [r[0] for r in resolved]
tax_subset["ncbi_species"] = [r[1] for r in resolved]
tax_subset["mapping_note"] = [r[2] for r in resolved]

# Flag novel/placeholder taxa
NOVEL_PREFIXES = ("UBA", "CAG-", "GUT_", "QAMM", "UMGS", "RUG", "SGB")
tax_subset["is_novel"] = tax_subset["gtdb_genus"].apply(
    lambda g: (
        any(g.startswith(p) for p in NOVEL_PREFIXES) or
        bool(re.match(r'^[A-Z]{2,4}[0-9]{4,}', g))
    )
)

print(f"\nTaxonomy parsing complete:")
print(f"  Direct (no remap):        {(tax_subset['mapping_note']=='direct').sum()}")
print(f"  Genus-mapped (GTDB→NCBI): {(tax_subset['mapping_note']=='genus_mapped').sum()}")
print(f"  Species override:         {(tax_subset['mapping_note']=='species_override').sum()}")
print(f"  Novel/placeholder:        {tax_subset['is_novel'].sum()}")
print(f"\nSample of genus-mapped OTUs:")
print(tax_subset[tax_subset["mapping_note"]=="genus_mapped"][
    ["OTU_ID","gtdb_genus","ncbi_genus"]
].head(8).to_string(index=False))


In [ ]:
# ── SCFA annotation (tiered matching) ──

POLYPHYLETIC_GENERA = {"Ruminococcus", "Clostridium", "Eubacterium"}

def annotate_scfa(ncbi_genus, ncbi_species, is_novel):
    """
    Look up SCFA producer status for a given taxon.
    Returns a dict with SCFA binary flags, variant info, match metadata.
    """
    result = {scfa: 0 for scfa in SCFA_TYPES}
    result.update({
        "butyrate_variant"   : None,
        "propionate_variant" : None,
        "match_tier"         : None,
        "match_taxon"        : None,
        "match_note"         : "",
    })

    if is_novel or ncbi_genus == "unknown":
        result["match_tier"]  = "uncharacterised"
        result["match_taxon"] = ncbi_genus
        result["match_note"]  = "Novel placeholder taxon — no Frolova entry"
        return result

    # --- Tier 1: species-level match ---
    if ncbi_species != "unknown" and " " in ncbi_species:
        sp_match = frolova[
            frolova["ncbi_species"].str.lower() == ncbi_species.lower()
        ]
        if len(sp_match) > 0:
            for scfa in SCFA_TYPES:
                if scfa in sp_match.columns:
                    result[scfa] = int(sp_match[scfa].mean() > 0)
            if "butyrate_variant" in sp_match.columns:
                variants = sp_match["butyrate_variant"].dropna().unique()
                result["butyrate_variant"] = (
                    "/".join(str(v) for v in variants) if len(variants) else None
                )
            if "propionate_variant" in sp_match.columns:
                variants = sp_match["propionate_variant"].dropna().unique()
                result["propionate_variant"] = (
                    "/".join(str(v) for v in variants) if len(variants) else None
                )
            result["match_tier"]  = "species"
            result["match_taxon"] = ncbi_species
            result["match_note"]  = f"n={len(sp_match)} strains in Frolova"
            return result

    # --- Tier 2: genus-level any-member ---
    if ncbi_genus in genus_lookup:
        entry = genus_lookup[ncbi_genus]
        for scfa in SCFA_TYPES:
            result[scfa] = entry.get(scfa, 0)
        n     = entry.get("n_genomes", "?")
        props = {s: entry.get(f"{s}_proportion", 0) for s in SCFA_TYPES}
        result["match_tier"]  = "genus"
        result["match_taxon"] = ncbi_genus
        result["match_note"]  = (
            f"Genus-level any-member (n={n} genomes); "
            f"proportions: " +
            ", ".join(f"{s}={props[s]:.2f}" for s in SCFA_TYPES)
        )
        if ncbi_genus in POLYPHYLETIC_GENERA:
            result["match_note"] += " [POLYPHYLETIC — interpret with caution]"
        return result

    # --- Tier 3: genus not in Frolova ---
    result["match_tier"]  = "no_match"
    result["match_taxon"] = ncbi_genus
    result["match_note"]  = "Genus not found in Frolova database"
    return result


# Apply annotation to all OTUs in scope
print("Annotating OTUs...")
annotations  = tax_subset.apply(
    lambda row: annotate_scfa(
        row["ncbi_genus"], row["ncbi_species"], row["is_novel"]
    ),
    axis=1
)
annot_df     = pd.DataFrame(annotations.tolist(), index=tax_subset.index)
tax_annotated = pd.concat([tax_subset, annot_df], axis=1)

print(f"\nSCFA annotation complete:")
print(f"  Tier 1 (species-level):   {(tax_annotated['match_tier']=='species').sum()}")
print(f"  Tier 2 (genus-level):     {(tax_annotated['match_tier']=='genus').sum()}")
print(f"  Tier 3 (no match):        {(tax_annotated['match_tier']=='no_match').sum()}")
print(f"  Tier 4 (uncharacterised): {(tax_annotated['match_tier']=='uncharacterised').sum()}")
print(f"\nProducer counts (all 9,612 OTUs in scope):")
for scfa in SCFA_TYPES:
    n = tax_annotated[scfa].sum()
    print(f"  {scfa:<14}: {n}")

print(f"\nSpot-check — known butyrate producers:")
check = tax_annotated[
    tax_annotated["ncbi_genus"].isin(["Faecalibacterium","Roseburia","Anaerostipes"])
][["OTU_ID","gtdb_genus","ncbi_genus","ncbi_species",
   "butyrate","propionate","match_tier","match_note"]].head(8)
print(check.to_string(index=False))

In [ ]:
# ── Build final annotation table ──

ANNOT_COLS = [
    "OTU_ID", "gtdb_genus", "gtdb_species", "ncbi_genus", "ncbi_species",
    "mapping_note", "is_novel", "match_tier", "match_taxon", "match_note",
    "butyrate", "propionate", "acetate", "formate", "lactate",
    "butyrate_variant", "propionate_variant",
]

# --- Stage 5: filter to significant OTUs only ---
print("--- Building Stage 5 annotation ---")
if "significant" in diff_ab.columns:
    sig_mask = diff_ab["significant"] == True
elif "padj" in diff_ab.columns:
    sig_mask = diff_ab["padj"] < 0.05
else:
    raise ValueError("Cannot find significance filter column in diff_ab. "
                     "Expected 'significant' or 'padj'.")

diff_ab_sig = diff_ab[sig_mask].copy()
print(f"Stage 5 total rows:       {len(diff_ab)}")
print(f"Stage 5 significant rows: {len(diff_ab_sig)}")
print(f"Unique significant OTUs:  {diff_ab_sig['OTU_ID'].nunique()}")

stage5_annot = diff_ab_sig.merge(
    tax_annotated[ANNOT_COLS],
    on="OTU_ID", how="left"
)
stage5_annot["source"] = "stage5"

# --- Stage 3: top 50 SHAP OTUs ---
print("\n--- Building Stage 3 annotation ---")
stage3_annot = feat_imp.merge(
    tax_annotated[ANNOT_COLS],
    on="OTU_ID", how="left"
)
stage3_annot["source"] = "stage3"
print(f"Stage 3 OTUs: {len(stage3_annot)}")

# --- Identify overlap ---
overlap_otus = set(diff_ab_sig["OTU_ID"]) & set(feat_imp["OTU_ID"])
print(f"\nOverlap (significant in Stage 5 AND top 50 in Stage 3): {len(overlap_otus)}")
if overlap_otus:
    print("Overlapping OTUs:")
    overlap_tax = tax_annotated[tax_annotated["OTU_ID"].isin(overlap_otus)][
        ["OTU_ID","ncbi_genus","ncbi_species","butyrate","propionate","acetate"]
    ]
    print(overlap_tax.to_string(index=False))

# Update source labels
stage5_annot.loc[stage5_annot["OTU_ID"].isin(overlap_otus), "source"] = "both"
stage3_annot.loc[stage3_annot["OTU_ID"].isin(overlap_otus), "source"] = "both"

# --- Combine into single table ---
# Stage 5 rows carry: fiber_group, log2FoldChange, padj, direction, group_name
# Stage 3 rows carry: shap_importance
# For overlap OTUs both are populated via the merge below

# Add shap_importance to stage5 rows where available
stage5_annot = stage5_annot.merge(
    feat_imp[["OTU_ID","shap_importance"]],
    on="OTU_ID", how="left"
)

# For stage3_only OTUs, fiber_group/log2FoldChange will be NaN — expected
stage3_only_ids   = set(feat_imp["OTU_ID"]) - set(diff_ab_sig["OTU_ID"])
stage3_only_rows  = stage3_annot[
    stage3_annot["OTU_ID"].isin(stage3_only_ids)
].copy()
print(f"\nStage 3 only (not in Stage 5 significant): {len(stage3_only_rows)}")

combined = pd.concat(
    [stage5_annot, stage3_only_rows],
    ignore_index=True, sort=False
)

# Sort: overlap first, then stage5, then stage3; within each by abs CLR diff
combined["abs_lfc"] = combined["log2FoldChange"].abs().fillna(0)
source_order        = {"both": 0, "stage5": 1, "stage3": 2}
combined["src_ord"] = combined["source"].map(source_order)
combined            = combined.sort_values(
    ["src_ord","abs_lfc"], ascending=[True, False]
).drop(columns=["abs_lfc","src_ord"])
combined            = combined.reset_index(drop=True)

print(f"\nCombined annotation table: {len(combined)} rows")
print(f"  source='both':   {(combined['source']=='both').sum()}")
print(f"  source='stage5': {(combined['source']=='stage5').sum()}")
print(f"  source='stage3': {(combined['source']=='stage3').sum()}")
print(f"\nFiber groups in final table: "
      f"{sorted(combined['fiber_group'].dropna().unique().astype(int))}")
print(f"\nAnnotation tier breakdown in combined table:")
print(combined["match_tier"].value_counts().to_string())

# Save
combined.to_csv(PATHS["out_scfa"], index=False)
print(f"\nSaved: {PATHS['out_scfa']}")

In [ ]:
# ── Nutrigenomic pathway summary (markdown) ──

def scfa_list(row):
    """Return list of SCFAs this OTU is annotated as producing."""
    return [s for s in SCFA_TYPES if row.get(s, 0) == 1]


lines = []
lines.append("# Nutrigenomic Pathway Summary")
lines.append(f"*Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n")

lines.append("## Methods note")
lines.append(
    "SCFA producer status was annotated using the Frolova et al. 2022 "
    "(*Front. Mol. Biosci.*) genome-scale reconstruction database (2,856 human gut "
    "bacterial genomes; 297 genera). Species-level matching was attempted first "
    "(Tier 1); genus-level any-member annotation was used as fallback (Tier 2). "
    "Effect sizes are reported as CLR differences (after − before, fiber vs control). "
    "HDAC and downstream gene linkages represent a literature-curated mechanistic "
    "hypothesis layer and are not derived from the sequencing data. "
    "Note: Bacteroides butyrate annotation reflects a single outlier genome "
    "(proportion=0.009) and should be interpreted cautiously.\n"
)

lines.append("---\n")
lines.append("## Fiber-enriched OTUs with SCFA annotation")
lines.append("*(Stage 5 significant, direction=up, Tier 1 or Tier 2 match only)*\n")

# Work from combined table — up-regulated, annotated, Stage 5 rows
up_otus = combined[
    (combined["direction"] == "up") &
    (combined["match_tier"].isin(["species", "genus"])) &
    (combined["source"].isin(["stage5", "both"]))
].copy()

print(f"Fiber-enriched annotated OTUs for summary: {len(up_otus)}")
print(f"Fiber groups: {sorted(up_otus['fiber_group'].dropna().unique().astype(int))}")

# Group by fiber group
for fg_id, fg_label in FIBER_GROUP_LABELS.items():
    fg_subset = up_otus[up_otus["fiber_group"] == fg_id].copy()
    if len(fg_subset) == 0:
        continue

    # Top 20 per fiber group by abs CLR diff for readability
    fg_subset = fg_subset.nlargest(20, "log2FoldChange")

    lines.append(f"### Group {fg_id} — {fg_label}")
    lines.append(f"*Top 20 fiber-enriched OTUs by CLR difference (of "
                 f"{(up_otus['fiber_group']==fg_id).sum()} total annotated)*\n")
    lines.append("| Taxon | CLR diff | Match tier | SCFA(s) | "
                 "Pathway variant | HDAC mechanism | Key target genes | Source |")
    lines.append("|-------|----------|------------|---------|"
                 "----------------|----------------|------------------|--------|")

    for _, row in fg_subset.iterrows():
        scfas = scfa_list(row)

        if not scfas:
            scfa_str  = "none detected"
            hdac_str  = "—"
            gene_str  = "—"
        else:
            scfa_str   = ", ".join(scfas)
            hdac_parts = []
            gene_parts = []
            for s in scfas:
                if s in HDAC_LINKAGE:
                    hdac_parts.append(HDAC_LINKAGE[s]["mechanism"])
                    gene_parts.extend(HDAC_LINKAGE[s]["target_genes"])
            # Deduplicate preserving order
            hdac_str = "; ".join(dict.fromkeys(hdac_parts))
            gene_str = ", ".join(dict.fromkeys(gene_parts))
            # Truncate for table readability
            if len(hdac_str) > 55:
                hdac_str = hdac_str[:52] + "..."
            if len(gene_str) > 45:
                gene_str = gene_str[:42] + "..."

        # Pathway variant string
        variant_parts = []
        if row.get("butyrate_variant") and "butyrate" in scfas:
            variant_parts.append(f"but:{row['butyrate_variant']}")
        if row.get("propionate_variant") and "propionate" in scfas:
            variant_parts.append(f"pro:{row['propionate_variant']}")
        variant_str = ", ".join(variant_parts) if variant_parts else "—"

        # Taxon label — prefer species, fall back to genus
        taxon = row.get("ncbi_species", "")
        if not taxon or taxon == "unknown":
            taxon = row.get("ncbi_genus", "unknown")

        clr   = row.get("log2FoldChange", float("nan"))
        clr_str = f"{clr:+.3f}" if pd.notna(clr) else "n/a"

        in_stage3 = "✓" if row["source"] == "both" else ""

        lines.append(
            f"| *{taxon}* | {clr_str} | {row['match_tier']} "
            f"| {scfa_str} | {variant_str} "
            f"| {hdac_str} | {gene_str} | {in_stage3} |"
        )
    lines.append("")

lines.append("---\n")
lines.append("## HDAC / gene target reference\n")
lines.append("| SCFA | Mechanism | Receptors | Target genes | Phenotype | References |")
lines.append("|------|-----------|-----------|--------------|-----------|------------|")
for scfa, entry in HDAC_LINKAGE.items():
    rec_str = ", ".join(entry["receptors"]) if entry["receptors"] else "—"
    lines.append(
        f"| {scfa} | {entry['mechanism']} | {rec_str} "
        f"| {', '.join(entry['target_genes'])} "
        f"| {entry['phenotype']} | {entry['references']} |"
    )

lines.append("\n---\n")
lines.append("## Annotation coverage summary\n")

total       = len(combined)
both_n      = (combined["source"] == "both").sum()
s5_n        = (combined["source"] == "stage5").sum()
s3_n        = (combined["source"] == "stage3").sum()
tier1_n     = (combined["match_tier"] == "species").sum()
tier2_n     = (combined["match_tier"] == "genus").sum()
nomatch_n   = (combined["match_tier"] == "no_match").sum()
novel_n     = (combined["match_tier"] == "uncharacterised").sum()

lines += [
    f"- Total rows in combined table: {total}",
    f"- OTUs in both Stage 3 and Stage 5: {both_n}",
    f"- Stage 5 only: {s5_n}",
    f"- Stage 3 only: {s3_n}",
    f"- Tier 1 (species-level match): {tier1_n}",
    f"- Tier 2 (genus-level match): {tier2_n}",
    f"- No match: {nomatch_n}",
    f"- Uncharacterised (novel taxa): {novel_n}",
    "",
]

lines.append("### Polyphyletic genus warnings\n")
poly_rows = combined[combined["match_note"].str.contains("POLYPHYLETIC", na=False)]
if len(poly_rows) > 0:
    seen = set()
    for _, row in poly_rows.iterrows():
        key = row["ncbi_genus"]
        if key not in seen:
            n = (poly_rows["ncbi_genus"] == key).sum()
            lines.append(f"- **{row['ncbi_genus']}** ({n} OTUs): "
                         f"{row['gtdb_genus']} collapsed to NCBI genus — "
                         f"interpret SCFA annotation with caution.")
            seen.add(key)
else:
    lines.append("- None detected.")

lines.append("")
lines.append("*End of report*")

md_text = "\n".join(lines)
with open(PATHS["out_summary_md"], "w", encoding="utf-8") as f:
    f.write(md_text)

print(f"\nSaved: {PATHS['out_summary_md']}")
print(f"Markdown length: {len(md_text)} characters, "
      f"{len(lines)} lines")

In [ ]:
# ── Figure: SCFA producer bar chart ──

SCFA_COLORS = {
    "butyrate"   : "#E07B39",
    "propionate" : "#5B8DB8",
    "acetate"    : "#6AAB6A",
}
SCFA_PLOT = ["butyrate", "propionate", "acetate"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "SCFA Producer Counts Among Fiber-Responsive OTUs by Fiber Group",
    fontsize=13, fontweight="bold", y=1.01
)

panel_configs = [
    {
        "title"  : "Stage 5 — Differential Abundance\n(significant, fiber-enriched)",
        "df"     : combined[
                       (combined["direction"] == "up") &
                       (combined["source"].isin(["stage5", "both"]))
                   ].copy(),
    },
    {
        "title"  : "Stage 3 — Top 50 SHAP OTUs\n(all, regardless of direction)",
        "df"     : combined[
                       combined["source"].isin(["stage3", "both"])
                   ].copy(),
    },
]

for ax, cfg in zip(axes, panel_configs):
    plot_df = cfg["df"]

    # Only include fiber groups that actually have data
    groups = sorted([
        g for g in FIBER_GROUP_LABELS.keys()
        if g in plot_df["fiber_group"].values
    ])

    if not groups:
        ax.set_title(cfg["title"], fontsize=10)
        ax.text(0.5, 0.5, "No data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11, color="#888888")
        continue

    n_scfa    = len(SCFA_PLOT)
    bar_width = 0.22
    x         = np.arange(len(groups))

    for i, scfa in enumerate(SCFA_PLOT):
        counts = []
        for fg in groups:
            fg_rows = plot_df[plot_df["fiber_group"] == fg]
            counts.append(int(fg_rows[scfa].sum()))

        offset = (i - n_scfa / 2 + 0.5) * bar_width
        bars   = ax.bar(
            x + offset, counts, bar_width,
            label=scfa.capitalize(),
            color=SCFA_COLORS[scfa],
            edgecolor="white", linewidth=0.6,
            zorder=3,
        )
        # Value labels on bars
        for bar, count in zip(bars, counts):
            if count > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3,
                    str(count),
                    ha="center", va="bottom",
                    fontsize=8, color="#333333"
                )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"Group {g}\n({FIBER_GROUP_LABELS[g]})" for g in groups],
        fontsize=8.5
    )
    ax.set_ylabel("Number of producer OTUs", fontsize=9)
    ax.set_title(cfg["title"], fontsize=9, pad=10)
    ax.legend(
        title="SCFA", fontsize=8, title_fontsize=8,
        loc="upper right", framealpha=0.7
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(PATHS["out_barplot"], dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {PATHS['out_barplot']}")

In [ ]:
# ── Figure: SCFA binary heatmap ──

MAX_ROWS_S5 = 30
MAX_ROWS_S3 = 50

SOURCE_COLORS = {
    "both"   : "#C8960C",
    "stage5" : "#5B8DB8",
    "stage3" : "#6AAB6A",
}
CMAP = mcolors.ListedColormap(["#F0F0F0", "#E07B39"])

def prep_heatmap_data(df, sort_col, max_rows, require_annotated=True):
    """
    Filter, sort and cap rows for heatmap.
    Returns (matrix_df, row_labels, source_tags).
    """
    if require_annotated:
        df = df[df["match_tier"].isin(["species", "genus"])].copy()
    df = df.dropna(subset=[sort_col])
    df = df.sort_values(sort_col, ascending=False).head(max_rows)

    # Row label: species or genus + fiber group
    def make_label(row):
        taxon = row.get("ncbi_species", "")
        if not taxon or taxon == "unknown":
            taxon = row.get("ncbi_genus", "unknown")
        # Truncate long names
        if len(taxon) > 32:
            taxon = taxon[:30] + ".."
        fg = row.get("fiber_group")
        fg_str = f" [G{int(fg)}]" if pd.notna(fg) else ""
        return f"{taxon}{fg_str}"

    labels      = df.apply(make_label, axis=1).tolist()
    source_tags = df["source"].tolist()
    matrix      = df[SCFA_TYPES].fillna(0).astype(int)

    return matrix, labels, source_tags


# --- Prepare data for each panel ---
s5_df = combined[
    (combined["direction"] == "up") &
    (combined["source"].isin(["stage5", "both"]))
].copy()

s3_df = combined[
    combined["source"].isin(["stage3", "both"])
].copy()

s5_matrix, s5_labels, s5_sources = prep_heatmap_data(
    s5_df, "log2FoldChange", MAX_ROWS_S5
)
s3_matrix, s3_labels, s3_sources = prep_heatmap_data(
    s3_df, "shap_importance", MAX_ROWS_S3
)

# --- Build figure ---
# Dynamic height based on number of rows
row_height = 0.32
s5_height  = max(6, len(s5_labels) * row_height + 2)
s3_height  = max(6, len(s3_labels) * row_height + 2)
fig_height = max(s5_height, s3_height)

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, fig_height),
    gridspec_kw={"wspace": 0.55}
)
fig.suptitle(
    "SCFA Producer Profile — Fiber-Responsive OTUs",
    fontsize=13, fontweight="bold", y=1.01
)

panel_data = [
    {
        "ax"      : axes[0],
        "matrix"  : s5_matrix,
        "labels"  : s5_labels,
        "sources" : s5_sources,
        "title"   : f"Stage 5 — Differential Abundance\n"
                    f"(top {len(s5_labels)} fiber-enriched, by CLR diff)",
    },
    {
        "ax"      : axes[1],
        "matrix"  : s3_matrix,
        "labels"  : s3_labels,
        "sources" : s3_sources,
        "title"   : f"Stage 3 — Top {len(s3_labels)} SHAP OTUs\n"
                    f"(by MDI importance)",
    },
]

for pd_cfg in panel_data:
    ax      = pd_cfg["ax"]
    matrix  = pd_cfg["matrix"]
    labels  = pd_cfg["labels"]
    sources = pd_cfg["sources"]

    if matrix.empty:
        ax.set_title(pd_cfg["title"], fontsize=9)
        ax.text(0.5, 0.5, "No annotated OTUs",
                ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color="#888888")
        continue

    sns.heatmap(
        matrix,
        ax=ax,
        cmap=CMAP,
        vmin=0, vmax=1,
        linewidths=0.4,
        linecolor="#CCCCCC",
        cbar=False,
        xticklabels=[s.capitalize() for s in SCFA_TYPES],
        yticklabels=labels,
        annot=False,
    )

    ax.set_title(pd_cfg["title"], fontsize=9, pad=10)
    ax.set_xlabel("SCFA type", fontsize=9, labelpad=6)
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=8.5, rotation=30)
    ax.tick_params(axis="y", labelsize=7, pad=2)

    # Colour y-axis labels by source
    for tick_label, src in zip(ax.get_yticklabels(), sources):
        tick_label.set_color(SOURCE_COLORS.get(src, "#333333"))
        tick_label.set_fontstyle("italic")

# --- Shared legend ---
legend_elements = [
    mpatches.Patch(facecolor="#E07B39", label="Producer"),
    mpatches.Patch(facecolor="#F0F0F0", edgecolor="#AAAAAA",
                   linewidth=0.8, label="Non-producer"),
    mpatches.Patch(color=SOURCE_COLORS["both"],   label="In both Stage 3 & 5"),
    mpatches.Patch(color=SOURCE_COLORS["stage5"], label="Stage 5 only"),
    mpatches.Patch(color=SOURCE_COLORS["stage3"], label="Stage 3 only"),
]
fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=5,
    fontsize=8.5,
    framealpha=0.85,
    bbox_to_anchor=(0.5, -0.03),
    title="Colour key",
    title_fontsize=8.5,
)

plt.savefig(PATHS["out_heatmap"], dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {PATHS['out_heatmap']}")
print(f"  Stage 5 panel: {len(s5_labels)} OTU rows")
print(f"  Stage 3 panel: {len(s3_labels)} OTU rows")




In [ ]:
# ── Stage 6 report ──

# Compute summary statistics for report
up_combined   = combined[combined["direction"] == "up"]
down_combined = combined[combined["direction"] == "down"]

sig_s5_otus   = combined[combined["source"].isin(["stage5","both"])]
s3_otus       = combined[combined["source"].isin(["stage3","both"])]

poly_rows     = combined[
    combined["match_note"].str.contains("POLYPHYLETIC", na=False)
]

report_lines = [
    "=" * 70,
    "Stage 6 — Nutrigenomic Pathway Layer — Run Report",
    f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 70,
    "",
    "INPUT SUMMARY",
    f"  Stage 5 total rows (all fiber groups):  {len(diff_ab)}",
    f"  Stage 5 significant rows:               {len(diff_ab_sig)}",
    f"  Stage 5 unique significant OTUs:        {diff_ab_sig['OTU_ID'].nunique()}",
    f"  Stage 3 top SHAP OTUs:                  {len(feat_imp)}",
    f"  Overlap (sig Stage 5 + top Stage 3):    {len(overlap_otus)}",
    f"  Frolova genomes loaded:                 {len(frolova)}",
    f"  Frolova genera covered:                 {len(genus_lookup)}",
    "",
    "ANNOTATION COVERAGE (combined table)",
    f"  Total rows:                             {len(combined)}",
    f"  source = both:                          {(combined['source']=='both').sum()}",
    f"  source = stage5:                        {(combined['source']=='stage5').sum()}",
    f"  source = stage3:                        {(combined['source']=='stage3').sum()}",
    "",
    f"  Tier 1 (species-level):                 {(combined['match_tier']=='species').sum()}",
    f"  Tier 2 (genus-level):                   {(combined['match_tier']=='genus').sum()}",
    f"  Tier 3 (no match):                      {(combined['match_tier']=='no_match').sum()}",
    f"  Tier 4 (uncharacterised/novel):         {(combined['match_tier']=='uncharacterised').sum()}",
    "",
    "PRODUCER COUNTS — ALL COMBINED ROWS",
]
for scfa in SCFA_TYPES:
    n = int(combined[scfa].sum()) if scfa in combined.columns else 0
    report_lines.append(f"  {scfa:<14}: {n}")

report_lines += [
    "",
    "PRODUCER COUNTS — FIBER-ENRICHED (direction=up, Stage 5 significant)",
]
for scfa in SCFA_TYPES:
    n = int(up_combined[scfa].sum()) if scfa in up_combined.columns else 0
    report_lines.append(f"  {scfa:<14}: {n}")

report_lines += [
    "",
    "PRODUCER COUNTS — FIBER-DEPLETED (direction=down, Stage 5 significant)",
]

for scfa in SCFA_TYPES:
    n = int(down_combined[scfa].sum()) if scfa in down_combined.columns else 0
    report_lines.append(f"  {scfa:<14}: {n}")

report_lines += [
    "",
    "PRODUCER COUNTS — STAGE 3 TOP 50 SHAP OTUs",
]
for scfa in SCFA_TYPES:
    n = int(s3_otus[scfa].sum()) if scfa in s3_otus.columns else 0
    report_lines.append(f"  {scfa:<14}: {n}")

report_lines += [
    "",
    "FIBER GROUP BREAKDOWN (fiber-enriched, annotated OTUs)",
]
for fg_id, fg_label in FIBER_GROUP_LABELS.items():
    fg_rows = up_combined[
        (up_combined["fiber_group"] == fg_id) &
        (up_combined["match_tier"].isin(["species","genus"]))
    ]
    if len(fg_rows) == 0:
        continue
    report_lines.append(f"  Group {fg_id} ({fg_label}): {len(fg_rows)} OTUs")
    for scfa in ["butyrate","propionate","acetate"]:
        n = int(fg_rows[scfa].sum())
        report_lines.append(f"    {scfa}: {n} producers")

report_lines += [
    "",
    "GTDB→NCBI MAPPING SUMMARY",
]
for note, count in tax_annotated["mapping_note"].value_counts().items():
    report_lines.append(f"  {note}: {count}")

report_lines += [""]
if len(poly_rows) > 0:
    report_lines.append(
        f"  WARNING: {len(poly_rows)} rows matched to polyphyletic genera."
    )
    seen = set()
    for _, row in poly_rows.iterrows():
        key = row["ncbi_genus"]
        if key not in seen:
            n = (poly_rows["ncbi_genus"] == key).sum()
            report_lines.append(
                f"    {key}: {n} OTUs — SCFA annotation is genus-level "
                f"average across divergent GTDB lineages"
            )
            seen.add(key)
else:
    report_lines.append("  No polyphyletic genus warnings.")

report_lines += [
    "",
    "METHODOLOGICAL NOTES",
    "  1. Effect sizes are CLR differences, not log2 fold changes.",
    "     Sign convention: positive = fiber-enriched, negative = fiber-depleted.",
    "  2. Genus-level SCFA annotation uses any-member threshold (proportion > 0).",
    "     Proportions are stored in match_note for transparency.",
    "  3. Bacteroides butyrate=1 reflects a single outlier genome (prop=0.009).",
    "     Interpret Bacteroides butyrate annotation with caution.",
    "  4. HDAC/gene linkages are literature-curated hypotheses, not data-derived.",
    "  5. Novel placeholder taxa (UBA*, CAG*, etc.) are excluded from annotation.",
    "",
    "OUTPUT FILES",
    f"  {PATHS['out_scfa']}",
    f"  {PATHS['out_summary_md']}",
    f"  {PATHS['out_barplot']}",
    f"  {PATHS['out_heatmap']}",
    "",
    "Stage 6 complete.",
    "=" * 70,
]

report_text = "\n".join(report_lines)
print(report_text)

with open(PATHS["out_report"], "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\nSaved: {PATHS['out_report']}")